 # Implementación Manual de K-Means desde Cero 
 **Inteligencia Artificial I — Actividad 2 — Grupo 5**
 **Integrantes:**
- Cardenas Torres Julian Camilo
- Vargas Catuche Jhon Alexander
- Vasquez Peña Juan Sebastian

**Algoritmo asignado:** K-Means (Aprendizaje No Supervisado / Clustering)
**Fundación Universitaria Los Libertadores**

---
## Tabla de contenido 
1. [Introducción](#1-introducción)
2. [Fundamentos Matemáticos](#2-fundamentos-matemáticos)
3. [Implementación desde Cero](#3-implementación-desde-cero)
4. [Prueba con Dataset Iris](#4-prueba-con-dataset-iris)
5. [Visualización del Funcionamiento](#5-visualización-del-funcionamiento)
6. [Validación de Correctitud](#6-validación-de-correctitud)
7. [Conclusiones](#7-conclusiones)
8. [Referencias](#8-referencias)



## 1. Introducción

### 1.1 ¿Qué es K-Means?

**K-Means** es uno de los algoritmos de **aprendizaje no supervisado** más utilizados en la industria. Su objetivo es **agrupar (clusterizar) un conjunto de datos en K grupos** (clusters) basándose en la similitud entre ellos, sin necesidad de etiquetas previas [1].

A diferencia de los algoritmos de aprendizaje supervisado (como árboles de decisión o regresión), K-Means **no recibe respuestas correctas durante el entrenamiento**. En su lugar, descubre patrones y estructuras ocultas en los datos por sí mismo.

El algoritmo fue propuesto formalmente por **Stuart Lloyd en 1957** dentro de Bell Labs, aunque no fue publicado hasta 1982. Posteriormente, **James MacQueen** acuñó el término "K-means" en 1967 [2].

### 1.2 ¿Para qué tipo de problemas sirve?

K-Means es ideal para problemas donde necesitamos **descubrir grupos naturales** dentro de los datos. Algunos ejemplos reales en la industria:

| Aplicación | Descripción |
|---|---|
| Segmentación de clientes | Agrupar consumidores con comportamientos similares para marketing personalizado |
| Compresión de imágenes | Reducir la cantidad de colores de una imagen agrupándolos |
| Análisis de documentos | Agrupar artículos o noticias por similitud temática |
| Bioinformática | Agrupar genes con patrones de expresión similares |
| Detección de anomalías | Identificar puntos que no pertenecen a ningún cluster |
| Sistemas de recomendación | Agrupar usuarios con gustos parecidos |

### 1.3 Idea intuitiva del algoritmo

Imagina que eres dueño de una pizzería y quieres abrir **3 nuevas sedes** para cubrir mejor a tus clientes. Tienes el mapa con la ubicación de tus mil clientes más frecuentes. ¿Dónde colocar las sedes para minimizar la distancia que recorre cada cliente?

K-Means resuelve este problema **automáticamente**:

1. Coloca 3 puntos al azar en el mapa (los **centroides**)
2. Asigna cada cliente al punto más cercano
3. Mueve cada punto al **centro promedio** de los clientes que tiene asignados
4. Repite los pasos 2 y 3 hasta que los puntos ya no se muevan

Cuando termina, esos 3 puntos representan las **ubicaciones óptimas** de las sedes, y los clientes están agrupados en 3 zonas claras.

### 1.4 Objetivos de este notebook

En este notebook implementaremos K-Means **desde cero**, usando únicamente **NumPy** y **Pandas** para operaciones matemáticas básicas, **sin recurrir a librerías de machine learning como scikit-learn**. Los objetivos son:

- Comprender el funcionamiento interno del algoritmo
- Implementar matemáticamente cada paso
- Probar el algoritmo en un dataset real (Iris)
- Visualizar el proceso de convergencia
- Validar la correctitud comparando con scikit-learn

## 2. Fundamentos Matemáticos
### 2.1 Definición formal del problema

## 2. Fundamentos Matemáticos

### 2.1 Definición formal del problema

Dado un conjunto de $n$ observaciones $X = \{x_1, x_2, ..., x_n\}$, donde cada $x_i \in \mathbb{R}^d$ (es decir, cada observación tiene $d$ características), K-Means busca particionar las observaciones en $K$ conjuntos $S = \{S_1, S_2, ..., S_K\}$ tal que se **minimice la suma de cuadrados intra-cluster** (también llamada *inercia* o WCSS - *Within-Cluster Sum of Squares*) [3]:

$$
\arg\min_{S} \sum_{k=1}^{K} \sum_{x \in S_k} \| x - \mu_k \|^2
$$

Donde:
- $K$ = número de clusters (definido por el usuario)
- $S_k$ = conjunto de puntos asignados al cluster $k$
- $\mu_k$ = centroide del cluster $k$ (vector promedio)
- $\| x - \mu_k \|^2$ = distancia euclidiana al cuadrado entre el punto $x$ y el centroide $\mu_k$

### 2.2 Distancia Euclidiana

La métrica fundamental de K-Means es la **distancia euclidiana**, que mide la "línea recta" entre dos puntos en el espacio $d$-dimensional:

$$
d(p, q) = \sqrt{\sum_{i=1}^{d}(p_i - q_i)^2}
$$

**Ejemplo en 2 dimensiones:**

Sean dos puntos $P = (3, 4)$ y $Q = (0, 0)$:

$$
d(P, Q) = \sqrt{(3-0)^2 + (4-0)^2} = \sqrt{9 + 16} = \sqrt{25} = 5
$$

> 📌 **Nota:** Para fines computacionales, K-Means usa la **distancia al cuadrado** (sin raíz), porque minimizar $d^2$ es equivalente a minimizar $d$, y evita el costo de calcular raíces cuadradas.

### 2.3 Cálculo del centroide

El centroide $\mu_k$ del cluster $k$ es simplemente el **vector promedio** (media aritmética) de todos los puntos asignados a ese cluster:

$$
\mu_k = \frac{1}{|S_k|} \sum_{x \in S_k} x
$$

Donde $|S_k|$ es la cantidad de puntos asignados al cluster $k$.

**Ejemplo:** Si al cluster 1 se le asignaron los puntos $(2,3)$, $(4,5)$ y $(6,7)$:

$$
\mu_1 = \frac{1}{3} \left( (2,3) + (4,5) + (6,7) \right) = \left( \frac{12}{3}, \frac{15}{3} \right) = (4, 5)
$$

### 2.4 Algoritmo de Lloyd (proceso de aprendizaje)

El proceso iterativo que ejecuta K-Means se conoce como **algoritmo de Lloyd** [4]. Consta de **4 fases** que se repiten hasta la convergencia:

#### **Fase 1: Inicialización**
Se eligen $K$ centroides iniciales. Existen varias estrategias:
- **Aleatoria**: seleccionar $K$ puntos al azar del dataset (la que implementaremos)
- **K-Means++**: estrategia más sofisticada que evita inicializaciones malas (la que usa scikit-learn por defecto)

#### **Fase 2: Asignación (Expectation step)**
Para cada punto $x_i$, se calcula su distancia a todos los centroides y se le asigna al **centroide más cercano**:

$$
S_k^{(t)} = \{ x_i : \| x_i - \mu_k^{(t)} \|^2 \leq \| x_i - \mu_j^{(t)} \|^2 \quad \forall j, 1 \leq j \leq K \}
$$

#### **Fase 3: Actualización (Maximization step)**
Cada centroide se recalcula como el promedio de los puntos asignados:

$$
\mu_k^{(t+1)} = \frac{1}{|S_k^{(t)}|} \sum_{x_i \in S_k^{(t)}} x_i
$$

#### **Fase 4: Verificación de convergencia**
El algoritmo se detiene cuando se cumple alguna de estas condiciones:
- Los centroides ya no cambian (o cambian menos que un umbral $\epsilon$)
- Se alcanza un número máximo de iteraciones
- Las asignaciones de los puntos no cambian

### 2.5 Pseudocódigo del algoritmo

```
ALGORITMO K-Means(X, K, max_iter, tol):
    
    Entrada:
        X        = matriz de datos (n filas, d columnas)
        K        = número de clusters
        max_iter = número máximo de iteraciones
        tol      = tolerancia para detectar convergencia

    1. Inicializar K centroides aleatorios eligiendo K puntos de X
    
    2. PARA t = 1 HASTA max_iter:
        
        a) Asignar cada punto al centroide más cercano:
           PARA cada punto x_i en X:
               calcular distancia de x_i a cada centroide
               asignar x_i al cluster del centroide más cercano
        
        b) Actualizar centroides:
           PARA cada cluster k:
               nuevo_centroide_k = promedio de puntos en cluster k
        
        c) Verificar convergencia:
           SI ||centroides_nuevos - centroides_viejos|| < tol:
               TERMINAR
    
    3. RETORNAR etiquetas y centroides finales
```

### 2.6 Complejidad computacional

La complejidad de K-Means por iteración es:

$$
O(n \cdot K \cdot d)
$$

Donde:
- $n$ = número de puntos
- $K$ = número de clusters
- $d$ = número de dimensiones (features)

Esto lo hace **muy escalable** comparado con otros algoritmos de clustering, lo cual explica su popularidad en la industria.

### 2.7 Ventajas y limitaciones

| Ventajas | Limitaciones |
|---|---|
| Simple de implementar e interpretar | Requiere definir $K$ manualmente |
| Computacionalmente eficiente | Sensible a la inicialización aleatoria |
| Escalable a datasets grandes | Asume clusters esféricos del mismo tamaño |
| Garantiza convergencia | Sensible a outliers |
| Resultados reproducibles (con semilla) | Requiere escalado de variables |